Import libraries and configure paths

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

project_dir = Path(
    r"C:\Users\U S E R\Desktop\ML Assigment\project"
)

results_dir = (
    project_dir / "data" / "processed" / "model_comparisons"
)

results_dir.mkdir(parents=True, exist_ok=True)

print("Gradient Boosting setup completed.")

Gradient Boosting setup completed.


Load processed training data

In [2]:
train_df = pd.read_csv(
    project_dir / "data" / "processed" /
    "math_train_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

assert X_train.notna().all().all()
assert y_train.notna().all()

print("Training records:", X_train.shape[0])
print("Features:", X_train.shape[1])

display(train_df.head())

Training records: 316
Features: 58


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
1,-0.5,-0.5,0.0,1.0,-1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,14
2,-0.5,-1.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0
3,-0.5,0.5,0.0,0.0,1.0,0.0,-1.0,1.0,0.5,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,9
4,-0.5,-0.5,-2.0,1.0,-1.0,0.0,-1.0,0.0,-0.5,0.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,8


Define Gradient Boosting experiments

In [3]:
gb_experiments = {
    "GB-1": {
        "n_estimators": 100,
        "learning_rate": 0.10,
        "max_depth": 3,
        "min_samples_leaf": 1
    },

    "GB-2": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_samples_leaf": 2
    },

    "GB-3": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 2,
        "min_samples_leaf": 2
    },

    "GB-4": {
        "n_estimators": 150,
        "learning_rate": 0.10,
        "max_depth": 2,
        "min_samples_leaf": 4
    },

    "GB-5": {
        "n_estimators": 300,
        "learning_rate": 0.10,
        "max_depth": 4,
        "min_samples_leaf": 2
    }
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

experiment_table = pd.DataFrame.from_dict(
    gb_experiments,
    orient="index"
)

experiment_table.index.name = "Experiment"

display(experiment_table.reset_index())

,Experiment,n_estimators,learning_rate,max_depth,min_samples_leaf
0,GB-1,100,0.10,3,1
1,GB-2,200,0.05,3,2
2,GB-3,300,0.03,2,2
3,GB-4,150,0.10,2,4
4,GB-5,300,0.10,4,2


Train and evaluate all five configurations

In [4]:
gb_results = []
gb_fold_results = []
gb_predictions = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, params in gb_experiments.items():

    print(f"Training {name}...")

    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise"
    )

    predictions = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv
    )

    gb_predictions[name] = predictions

    rounded_predictions = np.clip(
        np.rint(predictions),
        0,
        20
    ).astype(int)

    predicted_binary = (
        predictions >= 10
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    gb_results.append({
        "Experiment": name,

        "Estimators": params["n_estimators"],
        "Learning Rate": params["learning_rate"],
        "Max Depth": params["max_depth"],
        "Min Leaf": params["min_samples_leaf"],

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

    for fold in range(5):

        gb_fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train RMSE": np.sqrt(
                -scores["train_MSE"][fold]
            ),

            "Validation RMSE": np.sqrt(
                -scores["test_MSE"][fold]
            ),

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

gb_results_df = pd.DataFrame(gb_results)

gb_results_df = gb_results_df.sort_values(
    by=["Validation RMSE", "Validation MAE", "RMSE Gap"],
    ascending=[True, True, True]
).reset_index(drop=True)

gb_fold_df = pd.DataFrame(gb_fold_results)

print("All Gradient Boosting experiments completed.")

Training GB-1...
Training GB-2...
Training GB-3...
Training GB-4...
Training GB-5...
All Gradient Boosting experiments completed.


Regression error comparison table

In [5]:
regression_columns = [
    "Experiment",
    "Estimators",
    "Learning Rate",
    "Max Depth",
    "Min Leaf",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 65)
print("GRADIENT BOOSTING – REGRESSION ERROR COMPARISON")
print("=" * 65)

display(
    gb_results_df[regression_columns].round(4)
)

GRADIENT BOOSTING – REGRESSION ERROR COMPARISON


,Experiment,Estimators,Learning Rate,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,GB-5,300,0.10,4,2,0.0251,0.9385,0.0017,2.1579,0.0410,1.4427,0.9999,0.8950,1.4017
1,GB-2,200,0.05,3,2,0.4129,1.0217,0.3725,2.5320,0.6098,1.5463,0.9822,0.8807,0.9365
2,GB-1,100,0.10,3,1,0.4275,1.0320,0.3847,2.6126,0.6201,1.5721,0.9816,0.8765,0.9520
3,GB-3,300,0.03,2,2,0.6907,1.0170,1.1801,2.7162,1.0853,1.6032,0.9438,0.8725,0.5179
4,GB-4,150,0.10,2,4,0.6166,1.0579,0.8674,2.7729,0.9307,1.6250,0.9586,0.8690,0.6943


Grade prediction accuracy comparison

In [6]:
grade_columns = [
    "Experiment",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 65)
print("GRADIENT BOOSTING – GRADE ACCURACY COMPARISON")
print("=" * 65)

display(
    gb_results_df[grade_columns].round(2)
)

GRADIENT BOOSTING – GRADE ACCURACY COMPARISON


,Experiment,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,GB-5,43.99,68.99,89.56
1,GB-2,41.46,66.46,89.24
2,GB-1,39.87,67.09,89.24
3,GB-3,40.51,66.14,89.56
4,GB-4,36.71,64.87,87.97


Pass/fail classification comparison

In [7]:
classification_columns = [
    "Experiment",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 65)
print("GRADIENT BOOSTING – PASS/FAIL CLASSIFICATION")
print("=" * 65)

display(
    gb_results_df[classification_columns].round(4)
)

GRADIENT BOOSTING – PASS/FAIL CLASSIFICATION


,Experiment,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,GB-5,88.6076,0.9890,0.8404,0.9806,0.9086
1,GB-2,86.7089,0.9724,0.8263,0.9515,0.8934
2,GB-1,86.0759,0.9669,0.8216,0.9417,0.8883
3,GB-3,86.7089,0.9886,0.8122,0.9806,0.8918
4,GB-4,86.3924,0.9722,0.8216,0.9515,0.8906


Confusion matrix comparison table

In [9]:
confusion_columns = [
    "Experiment",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("=" * 65)
print("GRADIENT BOOSTING – CONFUSION MATRIX COMPARISON")
print("=" * 65)

display(
    gb_results_df[confusion_columns]
)

GRADIENT BOOSTING – CONFUSION MATRIX COMPARISON


,Experiment,TN,FP,FN,TP
0,GB-5,101,2,34,179
1,GB-2,98,5,37,176
2,GB-1,97,6,38,175
3,GB-3,101,2,40,173
4,GB-4,98,5,38,175


Overfitting and underfitting assessment

In [10]:
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

gb_results_df["Fitting Assessment"] = (
    gb_results_df.apply(
        assess_fitting,
        axis=1
    )
)

print(f"Baseline Validation RMSE: {baseline_rmse:.4f}")

display(
    gb_results_df[
        [
            "Experiment",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

Baseline Validation RMSE: 4.5635


,Experiment,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,GB-5,0.0410,1.4427,1.4017,Possible overfitting
1,GB-2,0.6098,1.5463,0.9365,Possible overfitting
2,GB-1,0.6201,1.5721,0.9520,Possible overfitting
3,GB-3,1.0853,1.6032,0.5179,Possible overfitting
4,GB-4,0.9307,1.6250,0.6943,Possible overfitting


Final complete comparison table

In [11]:
final_comparison = gb_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL GRADIENT BOOSTING REGRESSION COMPARISON")
print("=" * 70)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(
        final_comparison.round(4)
    )

FINAL GRADIENT BOOSTING REGRESSION COMPARISON


,Rank,Experiment,Estimators,Learning Rate,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP,Fitting Assessment
0,1,GB-5,300,0.10,4,2,0.0251,0.9385,0.0017,2.1579,0.0410,1.4427,0.9999,0.8950,1.4017,43.9873,68.9873,89.5570,88.6076,0.9890,0.8404,0.9806,0.9086,101,2,34,179,Possible overfitting
1,2,GB-2,200,0.05,3,2,0.4129,1.0217,0.3725,2.5320,0.6098,1.5463,0.9822,0.8807,0.9365,41.4557,66.4557,89.2405,86.7089,0.9724,0.8263,0.9515,0.8934,98,5,37,176,Possible overfitting
2,3,GB-1,100,0.10,3,1,0.4275,1.0320,0.3847,2.6126,0.6201,1.5721,0.9816,0.8765,0.9520,39.8734,67.0886,89.2405,86.0759,0.9669,0.8216,0.9417,0.8883,97,6,38,175,Possible overfitting
3,4,GB-3,300,0.03,2,2,0.6907,1.0170,1.1801,2.7162,1.0853,1.6032,0.9438,0.8725,0.5179,40.5063,66.1392,89.5570,86.7089,0.9886,0.8122,0.9806,0.8918,101,2,40,173,Possible overfitting
4,5,GB-4,150,0.10,2,4,0.6166,1.0579,0.8674,2.7729,0.9307,1.6250,0.9586,0.8690,0.6943,36.7089,64.8734,87.9747,86.3924,0.9722,0.8216,0.9515,0.8906,98,5,38,175,Possible overfitting


Select the best Gradient Boosting configuration

In [12]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_params = gb_experiments[best_name]

print("=" * 60)
print("BEST GRADIENT BOOSTING CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Parameters:", best_params)

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision: {best_result['Precision']:.4f}")
print(f"Recall   : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST GRADIENT BOOSTING CONFIGURATION
Experiment: GB-5
Parameters: {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 4, 'min_samples_leaf': 2}

REGRESSION METRICS
Validation RMSE : 1.4427
Validation MAE  : 0.9385
Validation R²   : 0.8950
RMSE Gap        : 1.4017

GRADE ACCURACY
Exact Accuracy  : 43.99%
Within ±1 Grade : 68.99%
Within ±2 Grades: 89.56%

PASS/FAIL CLASSIFICATION
Accuracy : 88.61%
Precision: 0.9890
Recall   : 0.8404
Specificity: 0.9806
F1 Score : 0.9086

CONFUSION MATRIX
TN: 101
FP: 2
FN: 34
TP: 179

FITTING ASSESSMENT
Possible overfitting
